Local Vector DB using Faiss.

In [1]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path


c:\Users\lj200\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [51]:
with open("vector_db/meta.json") as f:
    data = json.load(f)
    meta = data

print(meta)




{'0': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 1, 'distance_m': 0.0, 'text': 'Distance between turbine 1 and turbine 1 is 0.0 meters.'}, '1': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 2, 'distance_m': 325.8746444, 'text': 'Distance between turbine 1 and turbine 2 is 325.8746444 meters.'}, '2': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 3, 'distance_m': 688.7869458, 'text': 'Distance between turbine 1 and turbine 3 is 688.7869458 meters.'}, '3': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 4, 'distance_m': 1014.1309460000001, 'text': 'Distance between turbine 1 and turbine 4 is 1014.1309460000001 meters.'}, '4': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 5, 'distance_m': 1358.417176, 'text': 'Distance between turbine 1 and turbine 5 is 1358.417176 meters.'}, '5': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 6, 'distance_m': 1702.793077, 'text': 'Distance between turbine 1 and turbine 6 is 1702.79307

In [36]:
import re

def extract_metadata(text):
    # Pattern: "Distance between turbine X and turbine Y is Z meters."
    m = re.match(
        r"Distance between turbine (\d+) and turbine (\d+) is ([\d\.]+) meters\.",
        text
    )
    if m:
        a, b, dist = m.groups()
        return {
            "type": "turbine_distance",
            "turbine_a": int(a),
            "turbine_b": int(b),
            "distance_m": float(dist)
        }

    # Pattern: "Turbine X is Y km from the port."
    m = re.match(
        r"Turbine (\d+) is ([\d\.]+) km from the port\.",
        text
    )
    if m:
        tid, dist = m.groups()
        return {
            "type": "distance_from_port",
            "turbine_id": int(tid),
            "distance_km": float(dist)
        }

    # Pattern: "At wind speed X m/s, the turbine produces Y kW."
    m = re.match(
        r"At wind speed ([\d\.]+) m/s, the turbine produces ([\d\.]+) kW\.",
        text
    )
    if m:
        ws, power = m.groups()
        return {
            "type": "power_curve",
            "wind_speed": float(ws),
            "power_kw": float(power)
        }

    # Default fallback
    return {
        "type": "generic",
    }

meta_data = [extract_metadata(t) for t in training_texts]

for i in range(0, len(meta_data)):
    print(meta_data[i])



{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 1, 'distance_m': 0.0}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 2, 'distance_m': 325.8746444}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 3, 'distance_m': 688.7869458}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 4, 'distance_m': 1014.1309460000001}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 5, 'distance_m': 1358.417176}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 6, 'distance_m': 1702.793077}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 7, 'distance_m': 2061.3461169999996}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 8, 'distance_m': 2372.654365}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 9, 'distance_m': 2731.184334}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 10, 'distance_m': 622.6743017}
{'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 11, 'distance_m': 783.7610284}
{'type': 'turbine_distan

In [ ]:
new_meta = []
idx = 0
for i in range(0, len(meta_data)):
    new_meta.append({
        i: meta_data[i]
    })

print(new_meta[:5])


[{0: {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 1, 'distance_m': 0.0}}, {1: {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 2, 'distance_m': 325.8746444}}, {2: {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 3, 'distance_m': 688.7869458}}, {3: {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 4, 'distance_m': 1014.1309460000001}}, {4: {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 5, 'distance_m': 1358.417176}}]


799

In [53]:
with open("vector_db/meta.json", "w") as f:
    json.dump(meta, f, indent=2)


In [14]:
metadata = {}

for item in meta:
    for key, value in item.items():
        metadata[str(key)] = value


In [52]:
def build_text(meta):
    t = meta["type"]

    if t == "turbine_distance":
        return (
            f"Distance between turbine {meta['turbine_a']} "
            f"and turbine {meta['turbine_b']} "
            f"is {meta['distance_m']} meters."
        )

    if t == "distance_from_port":
        return (
            f"Turbine {meta['turbine_id']} is "
            f"{meta['distance_km']} kilometers from the port."
        )

    if t == "power_curve":
        return (
            f"At a wind speed of {meta['wind_speed']} meters per second, "
            f"the turbine produces {meta['power_kw']} kilowatts of power."
        )


    # fallback for unknown types
    return "No text available for this metadata entry."

print(meta)


for id, _meta in meta.items():
    print(id, _meta)
    text = build_text(_meta)
    meta[id]["text"] = text
    # print(f"ID: {id}, Text: {text}")

{'0': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 1, 'distance_m': 0.0, 'text': 'Distance between turbine 1 and turbine 1 is 0.0 meters.'}, '1': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 2, 'distance_m': 325.8746444, 'text': 'Distance between turbine 1 and turbine 2 is 325.8746444 meters.'}, '2': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 3, 'distance_m': 688.7869458, 'text': 'Distance between turbine 1 and turbine 3 is 688.7869458 meters.'}, '3': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 4, 'distance_m': 1014.1309460000001, 'text': 'Distance between turbine 1 and turbine 4 is 1014.1309460000001 meters.'}, '4': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 5, 'distance_m': 1358.417176, 'text': 'Distance between turbine 1 and turbine 5 is 1358.417176 meters.'}, '5': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 6, 'distance_m': 1702.793077, 'text': 'Distance between turbine 1 and turbine 6 is 1702.79307

In [ ]:
texts = []
ids = []


for doc_id, _meta in meta.items():
    texts.append(_meta["text"])
    ids.append(int(doc_id))


    


Everything below should work if executed 

In [6]:
import json
with open("vector_db/meta.json") as f:
    data = json.load(f)
    meta = data

print(meta)

{'0': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 1, 'distance_m': 0.0, 'text': 'Distance between turbine 1 and turbine 1 is 0.0 meters.'}, '1': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 2, 'distance_m': 325.8746444, 'text': 'Distance between turbine 1 and turbine 2 is 325.8746444 meters.'}, '2': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 3, 'distance_m': 688.7869458, 'text': 'Distance between turbine 1 and turbine 3 is 688.7869458 meters.'}, '3': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 4, 'distance_m': 1014.1309460000001, 'text': 'Distance between turbine 1 and turbine 4 is 1014.1309460000001 meters.'}, '4': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 5, 'distance_m': 1358.417176, 'text': 'Distance between turbine 1 and turbine 5 is 1358.417176 meters.'}, '5': {'type': 'turbine_distance', 'turbine_a': 1, 'turbine_b': 6, 'distance_m': 1702.793077, 'text': 'Distance between turbine 1 and turbine 6 is 1702.79307

In [7]:
texts = []
ids = []


for doc_id, _meta in meta.items():
    texts.append(_meta["text"])
    ids.append(int(doc_id))


In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path

import faiss

class FaissStore:
    def __init__(self, index_path, model_name="all-MiniLM-L6-v2"):
        self.index_path = Path(index_path)
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()

        if self.index_path.exists():
            self.index = faiss.read_index(str(self.index_path))
        else:
            base_index = faiss.IndexFlatL2(self.dim)
            self.index = faiss.IndexIDMap(base_index)  # <-- key change

    def embed(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return self.model.encode(texts, convert_to_numpy=True).astype("float32")

    def add(self, texts, ids):
        vectors = self.embed(texts)
        ids = np.array(ids, dtype="int64")
        self.index.add_with_ids(vectors, ids)
        self.save()

    def search(self, query, k=5):
        q = self.embed(query)
        distances, ids = self.index.search(q, k)
        return distances[0], ids[0]

    def save(self):
        faiss.write_index(self.index, str(self.index_path))

store = FaissStore("vector_index.faiss")
store.add(texts, ids)


## UPGRADED SENTENCE TRANSFORMER MODEL // TAKES AGES

# import faiss
# import numpy as np
# from sentence_transformers import SentenceTransformer
# from pathlib import Path

# class FaissStore:
#     def __init__(self, index_path, model_name="BAAI/bge-large-en-v1.5"):
#         self.index_path = Path(index_path)
#         self.model = SentenceTransformer(model_name)
#         self.dim = self.model.get_sentence_embedding_dimension()

#         if self.index_path.exists():
#             self.index = faiss.read_index(str(self.index_path))
#         else:
#             base_index = faiss.IndexFlatL2(self.dim)
#             self.index = faiss.IndexIDMap(base_index)

#     def embed(self, texts):
#         if isinstance(texts, str):
#             texts = [texts]
#         return self.model.encode(
#             texts,
#             convert_to_numpy=True,
#             normalize_embeddings=True
#         ).astype("float32")

#     def add(self, texts, ids):
#         vectors = self.embed(texts)
#         ids = np.array(ids, dtype="int64")
#         self.index.add_with_ids(vectors, ids)
#         self.save()

#     def search(self, query, k=5):
#         q = self.embed(query)
#         distances, ids = self.index.search(q, k)
#         return distances[0], ids[0]

#     def save(self):
#         faiss.write_index(self.index, str(self.index_path))


In [9]:
store = FaissStore("vector_index.faiss")
store.add(texts, ids)

dists, idxs = store.search("distance between turbine 2 and 4", k=5)
print(idxs)  # should be your custom IDs, e.g. [2, ...]

print(meta[str(idxs[0])]["text"])

dists, idxs = store.search("power generation at 10 m/s", k=5)
print(idxs)  # should be your custom IDs, e.g. [2, ...]

print(meta[str(idxs[0])]["text"])


[30 30 30 30 30]
Distance between turbine 2 and turbine 4 is 688.8541466 meters.
[741 741 741 741 741]
At a wind speed of 10.0 meters per second, the turbine produces 1730.0 kilowatts of power.
